# SDN IDS Enhancements Notebook

Reproduction-plus study targeting the improvements promised for *“Intrusion detection in software defined network using deep learning approaches”*. This notebook focuses on:
- richer feature usage (48+ feature space) and automated selection (RFE + SHAP)
- advanced sequence models (BiLSTM, attention-augmented CNN-LSTM)
- loss functions tailored for imbalance (Focal Loss, Class-Balanced Focal Loss)
- evaluation upgrades (K-fold CV, AUC-ROC, expanded metrics)



In [126]:
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import shap
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelBinarizer, LabelEncoder, StandardScaler
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42)
random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = Path('/home/kintopi/Desktop/CNET/research/InSDN_DatasetCSV')



In [127]:
print("=" * 60)
print("SDN IDS Enhancements - Runtime Context")
print("=" * 60)
print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA capability count: {torch.cuda.device_count()}")
print()



SDN IDS Enhancements - Runtime Context
Using device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU
CUDA capability count: 1



In [128]:
def load_datasets(file_names):
    """Load multiple CSV snapshots and concatenate."""
    frames = []
    for name in file_names:
        path = DATA_DIR / name
        if not path.exists():
            raise FileNotFoundError(f"Missing dataset: {path}")
        df = pd.read_csv(path)
        frames.append(df)
    data = pd.concat(frames, ignore_index=True)
    print(f"Loaded {len(frames)} files → {data.shape[0]} rows, {data.shape[1]-1} features")
    return data


def preprocess_dataframe(df, drop_cols=None):
    df = df.copy()
    if drop_cols:
        df = df[[col for col in df.columns if col not in drop_cols]]

    y_raw = df['Label']
    X_raw = df.drop(columns=['Label'])

    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(y_raw)

    scaler = StandardScaler()
    X = scaler.fit_transform(X_raw)

    feature_names = X_raw.columns.to_numpy()
    return X.astype(np.float32), y.astype(np.int64), feature_names, label_encoder, scaler



In [129]:
def run_rfe_selection(X, y, feature_names, top_k=24, estimator=None):
    estimator = estimator or RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        n_jobs=-1,
        class_weight='balanced_subsample',
        random_state=42
    )
    rfe = RFE(estimator=estimator, n_features_to_select=top_k, step=0.1)
    rfe.fit(X, y)
    selected_mask = rfe.support_
    selected_features = feature_names[selected_mask]
    print(f"RFE selected {len(selected_features)} features (top_k={top_k}).")
    return X[:, selected_mask], selected_features, rfe


def shap_feature_importance(model, X, feature_names, max_display=15):
    """Compute SHAP importance on a sample of X using the fitted tree model."""
    background_size = min(2000, X.shape[0])
    background = X[np.random.choice(X.shape[0], background_size, replace=False)]
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(background)

    if isinstance(shap_values, list):
        # Multi-class (old format): list of arrays, one per class
        # Each array: (n_samples, n_features)
        shap_abs = np.mean([np.abs(vals).mean(axis=0) for vals in shap_values], axis=0)
    else:
        # Check if 3D array (newer SHAP multi-class format)
        if shap_values.ndim == 3:
            # Shape: (n_samples, n_features, n_classes)
            # Average over both samples (axis=0) and classes (axis=2)
            shap_abs = np.abs(shap_values).mean(axis=(0, 2))
        else:
            # Shape: (n_samples, n_features) - binary or regression
            # Average over samples only
            shap_abs = np.abs(shap_values).mean(axis=0)
    
    # Ensure 1D
    shap_abs = np.atleast_1d(shap_abs).ravel()
    
    # Verify shapes match
    if shap_abs.shape[0] != feature_names.shape[0]:
        raise ValueError(
            f"Shape mismatch: shap_abs={shap_abs.shape}, "
            f"feature_names={feature_names.shape}, "
            f"X features={X.shape[1]}"
        )
    
    ranked_idx = np.argsort(shap_abs)[::-1]
    top_idx = ranked_idx[:max_display]
    
    imp_df = pd.DataFrame({
        'feature': feature_names[top_idx],
        'mean_abs_shap': shap_abs[top_idx]
    })
    
    return imp_df






In [130]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, input_dim, num_classes, hidden_size=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(
            input_size=1,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )
        self.bn = nn.BatchNorm1d(hidden_size * 2)
        self.fc1 = nn.Linear(hidden_size * 2, 128)
        self.drop = nn.Dropout(dropout)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        batch_size = x.size(0)
        x = x.view(batch_size, self.input_dim, 1)
        lstm_out, _ = self.lstm(x)
        pooled = torch.mean(lstm_out, dim=1)
        pooled = self.bn(pooled)
        out = F.relu(self.fc1(pooled))
        out = self.drop(out)
        return self.fc2(out)


class AttentionCNNLSTM(nn.Module):
    def __init__(self, input_dim, num_classes, conv_channels=128, lstm_hidden=128, dropout=0.3):
        super().__init__()
        self.input_dim = input_dim
        self.conv = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Conv1d(64, conv_channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(conv_channels),
            nn.ReLU()
        )
        self.lstm = nn.LSTM(
            input_size=conv_channels,
            hidden_size=lstm_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )
        self.attn_proj = nn.Sequential(
            nn.Linear(lstm_hidden * 2, lstm_hidden),
            nn.Tanh(),
            nn.Linear(lstm_hidden, 1)
        )
        self.fc = nn.Sequential(
            nn.Linear(lstm_hidden * 2, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        batch_size = x.size(0)
        x = x.view(batch_size, 1, self.input_dim)
        conv_out = self.conv(x)
        conv_out = conv_out.transpose(1, 2)
        lstm_out, _ = self.lstm(conv_out)
        attn_scores = self.attn_proj(lstm_out)
        attn_weights = torch.softmax(attn_scores, dim=1)
        context = torch.sum(attn_weights * lstm_out, dim=1)
        return self.fc(context)



In [131]:
def focal_loss(inputs, targets, gamma=2.0, alpha=None, reduction='mean'):
    ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=alpha)
    pt = torch.exp(-ce_loss)
    loss = (1 - pt) ** gamma * ce_loss
    if reduction == 'mean':
        return loss.mean()
    if reduction == 'sum':
        return loss.sum()
    return loss


def class_balanced_weights(labels, num_classes, beta=0.999):
    counts = np.bincount(labels, minlength=num_classes)
    effective_num = 1.0 - np.power(beta, counts)
    weights = (1.0 - beta) / np.maximum(effective_num, 1e-6)
    weights = weights / weights.sum() * num_classes
    return torch.tensor(weights, dtype=torch.float32)


def class_balanced_focal_loss(inputs, targets, num_classes, beta=0.999, gamma=1.5, reduction='mean'):
    weights = class_balanced_weights(targets.cpu().numpy(), num_classes=num_classes, beta=beta).to(inputs.device)
    return focal_loss(inputs, targets, gamma=gamma, alpha=weights, reduction=reduction)



In [132]:
def make_dataloader(X, y, batch_size=256, shuffle=True):
    tensor_x = torch.from_numpy(X)
    tensor_y = torch.from_numpy(y)
    dataset = TensorDataset(tensor_x, tensor_y)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


def compute_metrics(y_true, y_pred, y_prob, num_classes):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    lb = LabelBinarizer()
    lb.fit(range(num_classes))
    y_true_bin = lb.transform(y_true)
    if y_true_bin.shape[1] == 1:
        y_true_bin = np.hstack([1 - y_true_bin, y_true_bin])
    auc = roc_auc_score(y_true_bin, y_prob, average='macro', multi_class='ovr')
    return {
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'auc_macro': auc
    }



In [133]:
def build_loss_fn(name, num_classes, train_labels, gamma=2.0, beta=0.999):
    weights = class_balanced_weights(train_labels, num_classes).to(DEVICE)

    if name == 'ce':
        def loss_fn(logits, targets):
            return F.cross_entropy(logits, targets, weight=weights)
    elif name == 'focal':
        def loss_fn(logits, targets):
            return focal_loss(logits, targets, gamma=gamma, alpha=weights)
    elif name == 'cb_focal':
        def loss_fn(logits, targets):
            return class_balanced_focal_loss(logits, targets, num_classes=num_classes, beta=beta, gamma=gamma)
    else:
        raise ValueError(f"Unknown loss: {name}")
    return loss_fn


def train_one_fold(model, train_loader, val_loader, loss_fn, num_classes, epochs=40, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    best_state = None
    best_val_f1 = -np.inf

    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

        model.eval()
        val_preds, val_probs, val_true = [], [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(DEVICE)
                logits = model(xb)
                probs = F.softmax(logits, dim=1).cpu().numpy()
                preds = np.argmax(probs, axis=1)
                val_preds.extend(preds)
                val_probs.extend(probs)
                val_true.extend(yb.numpy())

        metrics = compute_metrics(np.array(val_true), np.array(val_preds), np.array(val_probs), num_classes)
        if metrics['f1'] > best_val_f1:
            best_val_f1 = metrics['f1']
            best_state = model.state_dict()

    model.load_state_dict(best_state)
    return model



In [134]:
def evaluate_model(model, data_loader, num_classes):
    model.eval()
    preds, probs, labels = [], [], []
    with torch.no_grad():
        for xb, yb in data_loader:
            xb = xb.to(DEVICE)
            logits = model(xb)
            prob = F.softmax(logits, dim=1).cpu().numpy()
            preds.extend(np.argmax(prob, axis=1))
            probs.extend(prob)
            labels.extend(yb.numpy())
    preds = np.array(preds)
    probs = np.array(probs)
    labels = np.array(labels)
    metrics = compute_metrics(labels, preds, probs, num_classes)
    report = classification_report(labels, preds, zero_division=0)
    cm = confusion_matrix(labels, preds)
    return metrics, report, cm



In [135]:
def run_kfold_pipeline(X, y, model_type='bilstm', loss_type='focal', n_splits=5, epochs=40, batch_size=256):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    num_classes = len(np.unique(y))
    fold_records = []
    overall_start = time.time()

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
        fold_start = time.time()
        print(f"\nFold {fold}/{n_splits} — model={model_type}, loss={loss_type}")
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        train_loader = make_dataloader(X_train, y_train, batch_size=batch_size, shuffle=True)
        val_loader = make_dataloader(X_val, y_val, batch_size=batch_size, shuffle=False)

        if model_type == 'bilstm':
            model = BiLSTMClassifier(input_dim=X.shape[1], num_classes=num_classes).to(DEVICE)
        elif model_type == 'attn_cnn_lstm':
            model = AttentionCNNLSTM(input_dim=X.shape[1], num_classes=num_classes).to(DEVICE)
        else:
            raise ValueError("model_type must be 'bilstm' or 'attn_cnn_lstm'")

        loss_fn = build_loss_fn(loss_type, num_classes, y_train)
        model = train_one_fold(model, train_loader, val_loader, loss_fn, num_classes, epochs=epochs)
        metrics, report, cm = evaluate_model(model, val_loader, num_classes)
        fold_records.append({'fold': fold, **metrics})
        print(report)
        print(cm)

        fold_time = time.time() - fold_start
        elapsed = time.time() - overall_start
        remaining_folds = n_splits - fold
        est_remaining = (elapsed / fold) * remaining_folds if fold > 0 else 0.0
        print(f"Fold {fold} duration: {fold_time:.2f}s | Elapsed: {elapsed:.2f}s | Est. remaining: {est_remaining:.2f}s")

    total_time = time.time() - overall_start
    metrics_df = pd.DataFrame(fold_records)
    summary = metrics_df.mean().to_dict()
    print("\nK-Fold Summary:")
    for k, v in summary.items():
        print(f"  {k}: {v:.4f}")
    print(f"Total training time: {total_time:.2f}s")
    return metrics_df



In [136]:
FILE_BUNDLES = ['48_features.csv']

stage_start = time.time()
df_raw = load_datasets(FILE_BUNDLES)
X, y, feature_names, label_encoder, scaler = preprocess_dataframe(df_raw)
print(f"Data prep time: {time.time() - stage_start:.2f}s")

print(f"Class distribution after encoding: {np.bincount(y)}")

stage_start = time.time()
X_selected, selected_features, rfe_model = run_rfe_selection(X, y, feature_names, top_k=32)
print(f"RFE selection time: {time.time() - stage_start:.2f}s")

stage_start = time.time()
shap_df = shap_feature_importance(rfe_model.estimator_, X_selected, selected_features, max_display=20)
print(f"SHAP computation time: {time.time() - stage_start:.2f}s")
print("Top SHAP-ranked features:\n", shap_df)

model_start = time.time()
metrics_bilstm = run_kfold_pipeline(X_selected, y, model_type='bilstm', loss_type='focal', epochs=30)
print(f"BiLSTM total training time: {time.time() - model_start:.2f}s")

model_start = time.time()
metrics_attn = run_kfold_pipeline(X_selected, y, model_type='attn_cnn_lstm', loss_type='cb_focal', epochs=30)
print(f"Attention CNN-LSTM total training time: {time.time() - model_start:.2f}s")

print("\nBiLSTM fold metrics:\n", metrics_bilstm)
print("\nAttention CNN-LSTM fold metrics:\n", metrics_attn)



Loaded 1 files → 343889 rows, 48 features
Data prep time: 1.34s
Class distribution after encoding: [ 68424 121942  53616   1405    192  98129    164     17]
RFE selected 32 features (top_k=32).
RFE selection time: 209.19s
SHAP computation time: 847.62s
Top SHAP-ranked features:
             feature  mean_abs_shap
0    Bwd Header Len       0.043201
1          Protocol       0.027711
2    Fwd Header Len       0.023340
3        Fwd Pkts/s       0.021729
4        Bwd Pkts/s       0.011639
5     Flow Duration       0.010888
6      Flow IAT Max       0.010671
7      Tot Fwd Pkts       0.010604
8       Flow Pkts/s       0.009562
9      Flow IAT Min       0.009385
10      Bwd IAT Min       0.007959
11    Flow IAT Mean       0.007199
12     Flow IAT Std       0.006633
13      Fwd IAT Max       0.006502
14     Pkt Size Avg       0.005877
15     Pkt Len Mean       0.005859
16  Bwd Pkt Len Max       0.005842
17      Fwd IAT Tot       0.005621
18  Bwd Pkt Len Std       0.005049
19      Bwd IAT Max 